# P19 review round 4 controls
G3s / G3s-pl: P18's one-sentence cut and placebo, 10 more guide runs each (15 per task in all). G3x / G3x-p / G3x-pc: escalating guide that flags notes whose use would depart from the instructions (guide only). G3n-nf: the open-prompt guide on the no-checker store. nfd: lessons written with the data profile and without the outcome; C1-nfd / E-nfd. A0h: no memory, hint-level task. Predictions: `quick_runs/predictions/P19.md`.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks_p12.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
import re
TASKS4 = ["sf_tf01_goal_imbalance", "sf_tf02_player_rep_norm", "sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage"]
SF6 = TASKS4 + ["sf_tf04_balance_torque_leakfree", "sf_tf06_metric_select_leakfree"]
TARGET = {"sf_tf01_goal_imbalance": ["checks_class_distribution", "uses_class_weight", "uses_resampling", "uses_stratified_split"],
          "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"], "sf_tf03_correlation_causation": ["column_x_included", "applies_causal_reasoning"],
          "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
WRONG_COL = {"sf_tf01_goal_imbalance": "assists", "sf_tf02_player_rep_norm": "height_in_cm",
             "sf_tf03_correlation_causation": "assists", "sf_tf05_credit_card_leakage": "reports"}
FIX = {"sf_tf05_credit_card_leakage": {164, 166, 167, 168, 169}, "sf_tf03_correlation_causation": {127, 129, 131, 132, 134, 135, 137}}
CONTRA3 = {130, 136}; WRONG_IDS = set(range(708, 720))
INSTR = r"task (says|asks|states|requires|instructs)|instruction|use all|all available|contradict"
T14 = [json.loads(l) for l in open(REPO / "quick_runs/results/P14/guide_trace.jsonl")]
T13 = [json.loads(l) for l in open(REPO / "quick_runs/results/P13/guide_trace.jsonl")]
print(Counter(t["arm"] for t in T14))

Counter({'G3o': 30, 'G3n': 30, 'G3o-p': 20, 'G3n-p': 20})


In [4]:
# Stage 1: fix lessons and wrong lessons, per guide arm (runs out of 5 or 20). P13 G3 / G3-p for reference.
def stage1(trs, arm):
    row = {"arm": arm}
    for t, ids in FIX.items():
        rs = [x for x in trs if x["arm"] == arm.split(" ")[0] and x["task"] == t]
        if not rs: continue
        row[t[:7] + " fix seen"] = sum(bool(set(x["seen"]) & ids) for x in rs)
        row[t[:7] + " fix shown"] = sum(bool(set(x["shown"]) & ids) for x in rs)
        row[t[:7] + " n"] = len(rs)
    if t := [x for x in trs if x["task"] == "sf_tf03_correlation_causation" and x["arm"] == arm.split(" ")[0]]:
        row["tf03 contra shown"] = sum(bool(set(x["shown"]) & CONTRA3) for x in t)
    return row
def wrong(trs, arm):
    rs = [x for x in trs if x["arm"] == arm]
    return {"arm": arm, "runs": len(rs), "wrong seen": sum(bool(set(x["seen"]) & WRONG_IDS) for x in rs),
            "wrong accepted": sum(bool(set(x["accepted"]) & WRONG_IDS) for x in rs), "wrong shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs),
            **{f"{t[:7]} shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs if x["task"] == t) for t in TASKS4}}
s1 = pd.DataFrame([stage1(T13, "G3"), stage1(T14, "G3o"), stage1(T14, "G3n")]).set_index("arm")
wr = pd.DataFrame([wrong(T13, "G3-p"), wrong(T14, "G3o-p"), wrong(T14, "G3n-p")]).set_index("arm")
print(s1.to_string()); wr

     sf_tf05 fix seen  sf_tf05 fix shown  sf_tf05 n  sf_tf03 fix seen  sf_tf03 fix shown  sf_tf03 n  tf03 contra shown
arm                                                                                                                   
G3                  5                  0          5                 5                  1          5                  4
G3o                 5                  0          5                 5                  1          5                  4
G3n                 5                  2          5                 5                  0          5                  5


,runs,wrong seen,wrong accepted,wrong shown,sf_tf01 shown,sf_tf02 shown,sf_tf03 shown,sf_tf05 shown
arm,,,,,,,,
G3-p,20,20,5,5,4,0,0,1
G3o-p,20,20,6,6,5,0,0,1
G3n-p,20,20,10,9,5,0,0,4


In [5]:
import sys, yaml
sys.path.insert(0, str(REPO / "quick_runs" / "p13")); sys.path.insert(0, str(REPO / "quick_runs" / "p12"))
import guide as GD
from scipy.stats import fisher_exact
fp = lambda a, na, b, nb: round(fisher_exact([[a, na - a], [b, nb - b]])[1], 4)
T18 = [json.loads(l) for l in open(REPO / "quick_runs/results/P18/guide_trace.jsonl")]
T19 = [json.loads(l) for l in open(REPO / "quick_runs/results/P19/guide_trace.jsonl")]
print(Counter(t["arm"] for t in T19))
# 1. One-sentence cut against placebo, per task, P18 + P19 runs
cut = []
for task, ids in FIX.items():
    row = {"task": task[:7]}
    for arm in ("G3s", "G3s-pl"):
        rs = [t for t in T18 + T19 if t["arm"] == arm and t["task"] == task]
        row[arm + " fix shown"] = sum(bool(set(t["shown"]) & ids) for t in rs); row[arm + " n"] = len(rs)
    row["p"] = fp(row["G3s fix shown"], row["G3s n"], row["G3s-pl fix shown"], row["G3s-pl n"])
    cut.append(row)
cut = pd.DataFrame(cut); cut

Counter({'G3s': 20, 'G3s-pl': 20, 'G3x': 20, 'G3x-p': 20, 'G3x-pc': 20, 'G3n-nf': 20})


,task,G3s fix shown,G3s n,G3s-pl fix shown,G3s-pl n,p
0,sf_tf05,14,15,4,15,0.0005
1,sf_tf03,15,15,1,15,0.0000


In [6]:
# 2. Escalating guide: runs (of 5 per task) in which a fix / wrong lesson was flagged for the user or shown to the agent
rows = []
for t in T19:
    if not t["arm"].startswith("G3x"): continue
    f, s = set(t["flagged"]), set(t["shown"]); fx = FIX.get(t["task"], set())
    rows.append({"arm": t["arm"], "task": t["task"][:7], "fix flagged": bool(f & fx), "fix shown": bool(s & fx),
                 "wrong flagged": bool(f & WRONG_IDS), "wrong shown": bool(s & WRONG_IDS), "n flagged": len(f)})
esc = pd.DataFrame(rows)
esc_t = esc.groupby(["arm", "task"])[["fix flagged", "fix shown", "wrong flagged", "wrong shown"]].sum()
print(esc.groupby("arm")[["fix flagged", "fix shown", "wrong flagged", "wrong shown", "n flagged"]].sum().to_string())
esc_t

        fix flagged  fix shown  wrong flagged  wrong shown  n flagged
arm                                                                  
G3x              10          0              0            0         69
G3x-p            10          0             18            2         83
G3x-pc            5          0              1           20         56


fix flagged  fix shown  wrong flagged  wrong shown
arm    task                                                       
G3x    sf_tf01            0          0              0            0
       sf_tf02            0          0              0            0
       sf_tf03            5          0              0            0
       sf_tf05            5          0              0            0
G3x-p  sf_tf01            0          0              3            2
       sf_tf02            0          0              5            0
       sf_tf03            5          0              5            0
       sf_tf05            5          0              5            0
G3x-pc sf_tf01            0          0              0            5
       sf_tf02            0          0              0            5
       sf_tf03            5          0              1            5
       sf_tf05            0          0              0            5

In [7]:
# Reasons on flagged lessons (examples)
for t in T19:
    if not t["arm"].startswith("G3x") or t["run"] != "seed0": continue
    for r in t["rounds"]:
        for j in r.get("judgements") or []:
            if j.get("conflict") is True:
                try: i = int(j["id"])
                except (TypeError, ValueError, KeyError): continue
                kind = "fix" if i in FIX.get(t["task"], set()) else "wrong" if i in WRONG_IDS else "other"
                print(f"{t['arm']} {t['task'][:7]} {kind} [{i}]: {str(j.get('reason'))[:200]}")

G3x sf_tf01 other [109]: The task specifies a split of 80/20 with random_state=42. Adding 'stratify' changes the resulting training and validation sets, which may conflict with the intended split for reproducibility or benchm
G3x sf_tf01 other [115]: Same as 109; stratification may conflict with the specific split instructions provided.
G3x sf_tf01 other [112]: Same as 109; stratification may conflict with the specific split instructions provided.
G3x sf_tf03 fix [135]: The task explicitly instructs to 'Keep only numerical feature columns'. This note suggests using domain-driven selection to exclude certain numerical columns, which directly conflicts with the prompt'
G3x sf_tf03 fix [137]: Like id 135, this suggests excluding features based on causal reasoning, which conflicts with the instruction to 'Keep only numerical feature columns'.
G3x sf_tf03 other [128]: This note suggests excluding non-descriptive column names (like 'column_x'). Since the prompt explicitly asks for numerical 

In [8]:
# 3. Open prompt on the no-checker store; 4. labels of the write-time-probe (nfd) lessons
NF, NFD = GD.load_store("nf"), GD.load_store("nfd")
def ids(store, labfile, label):
    LAB = yaml.safe_load(open(REPO / labfile))
    return {i for i, L in enumerate(store) if L.get("nf") and L["source_task"] in LAB
            and f"{L['source_run']}/{L['k']}" in (LAB[L["source_task"]][label] or [])}
REINF = ids(NF, "quick_runs/p17/labels_nf.yaml", "reinforcing")
g3nnf = pd.DataFrame([{"task": t["task"][:7], "reinf shown": bool(set(t["shown"]) & REINF)} for t in T19 if t["arm"] == "G3n-nf"]).groupby("task").sum()
LABD = yaml.safe_load(open(REPO / "quick_runs/p19/labels_nfd.yaml"))
nfd_labels = {t[:7]: {k: len(v or []) for k, v in d.items()} for t, d in LABD.items()}
nfd_n = Counter(L["source_task"][:7] for L in NFD if L.get("probe"))
print("nfd lessons:", sum(nfd_n.values()), dict(nfd_n)); print("labels:", nfd_labels)
g3nnf

nfd lessons: 119 {'sf_tf01': 15, 'sf_tf02': 15, 'sf_tf03': 15, 'sf_tf04': 29, 'sf_tf05': 15, 'sf_tf06': 30}
labels: {'sf_tf01': {'fix': 2, 'reinforcing': 0}, 'sf_tf02': {'fix': 0, 'reinforcing': 5}, 'sf_tf03': {'fix': 5, 'reinforcing': 2}, 'sf_tf05': {'fix': 0, 'reinforcing': 0}}


,reinf shown
task,
sf_tf01,0
sf_tf02,5
sf_tf03,5
sf_tf05,0


In [9]:
# Stage 2: agent runs. Targeted checks on the 4 failing tasks (passes out of 5), P14 G3n next to P12/P13 arms
def load(arm):
    recs = [json.loads(l) for l in open(RUNS / arm / "records.jsonl")]
    recs = [r for r in recs if r["task_id"] in SF6]
    for r in recs:
        ev = [json.loads(l) for l in open(RUNS / arm / r["task_id"] / r["run"] / "events.jsonl")]
        calls = [e for e in ev if e["event"] == "llm_call" and e.get("usage")]
        r["sum_tokens"] = sum(c["usage"]["prompt_tokens"] + c["usage"]["completion_tokens"] for c in calls)
    return recs
ARMS = ["P12_A0", "P12_C1", "P13_E", "P17_C1nf", "P17_Enf", "P19_C1nfd", "P19_Enfd", "P19_A0h"]
NAME = dict(zip(ARMS, ["A0 none", "C1 own", "E emb", "C1-nf", "E-nf", "C1-nfd", "E-nfd", "A0h hint"]))
recs = {a: load(a) for a in ARMS}
print({a: (len(v), dict(Counter(r["stop"] for r in v))) for a, v in recs.items()})
rows = []
for t, checks in TARGET.items():
    for c in checks:
        rows.append({"task": t[:7], "check": c, **{NAME[a]: sum(r["checks"].get(c) is True for r in recs[a] if r["task_id"] == t) for a in ARMS}})
targeted = pd.DataFrame(rows)
pooled = targeted[[NAME[a] for a in ARMS]].sum() / 40
print("pooled:", pooled.round(3).to_dict()); targeted

{'P12_A0': (30, {'final_answer': 30}), 'P12_C1': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_E': (30, {'final_answer': 29, 'max_steps': 1}), 'P17_C1nf': (30, {'final_answer': 30}), 'P17_Enf': (30, {'final_answer': 30}), 'P19_C1nfd': (30, {'final_answer': 30}), 'P19_Enfd': (30, {'final_answer': 30}), 'P19_A0h': (20, {'final_answer': 20})}
pooled: {'A0 none': 0.075, 'C1 own': 0.75, 'E emb': 0.475, 'C1-nf': 0.025, 'E-nf': 0.025, 'C1-nfd': 0.6, 'E-nfd': 0.15, 'A0h hint': 0.725}


,task,check,A0 none,C1 own,E emb,C1-nf,E-nf,C1-nfd,E-nfd,A0h hint
0,sf_tf01,checks_class_distribution,0,5,5,0,1,4,2,5
1,sf_tf01,uses_class_weight,0,5,5,0,0,5,1,0
2,sf_tf01,uses_resampling,0,0,0,0,0,0,0,0
3,sf_tf01,uses_stratified_split,3,5,5,1,0,5,3,5
4,sf_tf02,handles_unit_suffix_in_height,0,4,4,0,0,0,0,5
5,sf_tf03,column_x_included,0,5,0,0,0,5,0,5
6,sf_tf03,applies_causal_reasoning,0,5,0,0,0,5,0,4
7,sf_tf05,used_leakage_features,0,1,0,0,0,0,0,5


In [10]:
# Exposure of the nfd store: runs (of 5) shown a fix / reinforcing nfd lesson
FIXD, REINFD = ids(NFD, "quick_runs/p19/labels_nfd.yaml", "fix"), ids(NFD, "quick_runs/p19/labels_nfd.yaml", "reinforcing")
TN = [json.loads(l) for l in open(REPO / "quick_runs/results/P19/trace_nfd.jsonl")]
expo = pd.DataFrame([{"arm": t["arm"], "task": t["task"][:7], "fix shown": bool(set(t["shown"]) & FIXD),
                      "reinf shown": bool(set(t["shown"]) & REINFD)} for t in TN if t["task"] in TASKS4]).groupby(["arm", "task"]).sum()
expo

fix shown  reinf shown
arm    task                           
C1-nfd sf_tf01          5            0
       sf_tf02          0            5
       sf_tf03          5            5
       sf_tf05          0            0
E-nfd  sf_tf01          0            0
       sf_tf02          0            0
       sf_tf03          0            5
       sf_tf05          0            0

In [11]:
# Harm and exact tests
sc = pd.DataFrame([{"arm": NAME[a], "task": r["task_id"], "score": r["score"]} for a in ARMS for r in recs[a]])
scores = sc.pivot_table(index="task", columns="arm", values="score", aggfunc="mean").round(4)
passed = (pooled * 40).round().astype(int)
fisher = {"C1-nf vs C1-nfd": fp(passed["C1-nf"], 40, passed["C1-nfd"], 40), "C1-nfd vs C1": fp(passed["C1-nfd"], 40, passed["C1 own"], 40),
          "E-nf vs E-nfd": fp(passed["E-nf"], 40, passed["E-nfd"], 40), "A0 vs A0h": fp(passed["A0 none"], 40, passed["A0h hint"], 40),
          **{f"cut vs placebo {r.task}": r.p for r in cut.itertuples()}}
print(fisher); scores

{'C1-nf vs C1-nfd': np.float64(0.0), 'C1-nfd vs C1': np.float64(0.2324), 'E-nf vs E-nfd': np.float64(0.1084), 'A0 vs A0h': np.float64(0.0), 'cut vs placebo sf_tf05': 0.0005, 'cut vs placebo sf_tf03': 0.0}


arm,A0 none,A0h hint,C1 own,C1-nf,C1-nfd,E emb,E-nf,E-nfd
task,,,,,,,,
sf_tf01_goal_imbalance,7.603000e-01,7.804000e-01,7.765000e-01,7.860000e-01,7.715000e-01,7.713000e-01,7.243000e-01,7.879000e-01
sf_tf02_player_rep_norm,4.664201e+06,4.650909e+06,4.650909e+06,4.664201e+06,4.664201e+06,4.650909e+06,4.664201e+06,4.664201e+06
sf_tf03_correlation_causation,2.769462e+07,4.774348e+06,4.755266e+06,2.769458e+07,4.699080e+06,2.769464e+07,2.769458e+07,2.769464e+07
sf_tf04_balance_torque_leakfree,1.000000e+00,NaN,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
sf_tf05_credit_card_leakage,9.659000e-01,8.220000e-01,8.220000e-01,9.659000e-01,9.659000e-01,9.144000e-01,9.659000e-01,9.659000e-01
sf_tf06_metric_select_leakfree,9.798000e-01,NaN,9.806000e-01,9.797000e-01,9.799000e-01,9.796000e-01,9.803000e-01,9.805000e-01


In [12]:
summary = {"cut_per_task": cut.to_dict("records"), "escalation": esc_t.reset_index().to_dict("records"),
  "escalation_totals": esc.groupby("arm")[["fix flagged", "fix shown", "wrong flagged", "wrong shown"]].sum().reset_index().to_dict("records"),
  "g3n_nf_reinf_shown": g3nnf.reset_index().to_dict("records"), "nfd_lessons": dict(nfd_n), "nfd_labels": nfd_labels,
  "nfd_exposure": expo.reset_index().to_dict("records"),
  "runs": {a: len(v) for a, v in recs.items()}, "stops": {a: dict(Counter(r["stop"] for r in v)) for a, v in recs.items()},
  "targeted": targeted.to_dict("records"), "targeted_pooled": pooled.round(3).to_dict(), "fisher_p": fisher,
  "score_mean": scores.to_dict("index")}
json.dump(summary, open(REPO / "quick_runs/results/P19/summary.json", "w"), indent=1, default=str)
"done" 

'done'